# Address Risk Walkthrough

Offline notebook skeleton for reading analytical data from ClickHouse, S3/Parquet, or Kafka replay topics. This notebook must not connect to blockchain RPC endpoints or participate in crawler hot paths.

In [ ]:
from analytics.clients.clickhouse import ClickHouseClient
from analytics.features.address_profile import profile_address
from analytics.features.token_flow import summarize_token_flow, detect_large_transfers
from analytics.features.risk_features import extract_risk_features
from analytics.models.risk_score import score_address

client = ClickHouseClient()

In [ ]:
params = {
    "tenant_id": "",
    "chain": "ethereum",
    "network": "mainnet",
    "address": "0x0000000000000000000000000000000000000000",
    "lookback_days": 30,
}

sql = """
SELECT tx_hash, token_address, token_symbol, from_address, to_address, amount_usd, amount_usd_value
FROM telemetry_fabric.chain_token_transfers FINAL
WHERE tenant_id = {tenant_id:String}
  AND chain = {chain:String}
  AND network = {network:String}
  AND (from_address = {address:String} OR to_address = {address:String})
  AND block_timestamp >= now64(3, 'UTC') - toIntervalDay({lookback_days:UInt32})
  AND reorged = 0
"""
transfers = client.query_dataframe(sql, params)
transfers.head()

In [ ]:
profile = profile_address(transfers, params["address"])
flow = summarize_token_flow(transfers, params["address"])
large = detect_large_transfers(transfers, min_amount_usd="100000")
features = extract_risk_features(profile, flow)
score = score_address(features)
profile, flow, len(large), score

## S3/Parquet Skeleton

Use `S3ParquetReader` for offline snapshots or curated datasets.

In [ ]:
from analytics.clients.s3 import S3ParquetReader

reader = S3ParquetReader()
# df = reader.scan_polars("s3://telemetry-fabric-curated/blockchain/token_transfers/**/*.parquet")
# df.filter(...).collect()